# 46. Generative models: autoencoders, GANs, and diffusion

**Level:** Advanced

**Prerequisites:** 27-29: neural training and image concepts.

[Course index](00_START_HERE.ipynb) | [Previous](45_Quantization_KV_Cache_and_Inference_Cost.ipynb) | [Next](47_Multimodal_Models_and_Contrastive_Learning.ipynb)

**Topics:** Generative AI, Autoencoder, VAE, GAN, Diffusion, Denoising, Latent space.

**Matching script:** `examples\46_Generative_Models_Autoencoders_and_Diffusion.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Generative models learn distributions or generation procedures, not just class labels.
An autoencoder compresses and reconstructs data; a plain autoencoder is not automatically a
reliable sampler of new examples. A VAE regularizes a probabilistic latent representation and
balances reconstruction with a distributional penalty.

A GAN trains a generator against a discriminator in an adversarial objective. Training can be
unstable and suffer mode collapse. Diffusion models learn to reverse a progressive noising
process, often through noise or related target prediction with conditioning.

These families differ in objectives, sampling, coverage, and resource requirements. Generated
media can contain artifacts, biases, or memorized material. Evaluate data rights, provenance,
privacy, and misuse risks rather than treating visual plausibility as sufficient quality.

The exercise shows a forward noising step and an oracle algebraic reconstruction using the
known injected noise. It deliberately does not claim to train a diffusion generator. A real
denoiser must estimate the unknown noise or another parameterization from noisy observations.

## Mathematics or data contract

A forward step is x_t = sqrt(alpha_bar)*x_0 + sqrt(1-alpha_bar)*epsilon.

## Important limitation

Oracle inversion with known noise is not learned denoising or image generation.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Inspect the forward diffusion equation

**Purpose:** Understand signal/noise mixing before training a model.

### Read the code in this order
1. Create a small clean signal and seeded Gaussian noise.
2. Mix with an explicit alpha_bar.
3. Invert using the known injected noise.
4. Assert exact reconstruction within numerical tolerance and state the oracle limitation.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
clean shape: (12,) noisy differs: True
oracle reconstruction error: 0.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np

clean = np.linspace(-1, 1, 12)
noise = np.random.default_rng(42).normal(size=clean.shape)
alpha_bar = 0.4
noisy = np.sqrt(alpha_bar) * clean + np.sqrt(1 - alpha_bar) * noise
oracle_reconstruction = (noisy - np.sqrt(1 - alpha_bar) * noise) / np.sqrt(alpha_bar)
np.testing.assert_allclose(oracle_reconstruction, clean, atol=1e-12)
assert not np.allclose(noisy, clean)
print("clean shape:", clean.shape, "noisy differs:", not np.allclose(noisy, clean))
print("oracle reconstruction error:", round(float(np.abs(oracle_reconstruction - clean).max()), 12))

clean shape: (12,) noisy differs: True
oracle reconstruction error: 0.0


### Interpret example 1

The result should demonstrate this purpose: **Understand signal/noise mixing before training a model.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Train a denoising network on synthetic signals

**Purpose:** Distinguish a learned denoiser from the oracle inversion example.

### Read the code in this order
1. Generate low-dimensional clean signals embedded in a larger vector.
2. Add independent Gaussian noise.
3. Train a small network using only the training split.
4. Compare held-out reconstruction MSE against the noisy-input baseline.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
held-out denoising MSE: 0.0617 noisy baseline: 0.255
a learned denoiser, not a complete diffusion sampler
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(42)
latent = torch.randn(300, 1)
clean = latent @ torch.tensor([[1., 0.5, -0.5, -1.]])
noisy = clean + 0.5 * torch.randn_like(clean)
model = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 4))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
for _ in range(140):
    optimizer.zero_grad()
    loss = ((model(noisy[:220]) - clean[:220]) ** 2).mean()
    loss.backward()
    optimizer.step()
model.eval()
with torch.inference_mode():
    error = ((model(noisy[220:]) - clean[220:]) ** 2).mean().item()
    baseline = ((noisy[220:] - clean[220:]) ** 2).mean().item()
assert error < baseline
print("held-out denoising MSE:", round(error, 4), "noisy baseline:", round(baseline, 4))
print("a learned denoiser, not a complete diffusion sampler")

held-out denoising MSE: 0.0617 noisy baseline: 0.255
a learned denoiser, not a complete diffusion sampler


### Interpret example 2

The result should demonstrate this purpose: **Distinguish a learned denoiser from the oracle inversion example.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Train a tiny noise predictor on scalar Gaussian examples and report held-out error versus a zero-noise baseline.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Generative AI` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A forward step is x_t = sqrt(alpha_bar)*x_0 + sqrt(1-alpha_bar)*epsilon.
- Name the boundary of the demonstration: Oracle inversion with known noise is not learned denoising or image generation.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/2006.11239`
- `https://arxiv.org/abs/1312.6114`
- `https://arxiv.org/abs/1406.2661`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.